# SPXW 2026 YTD 热门期权与滚动价格

读取受管的 Massive OPRA Day Aggregates Parquet。单个 SPXW 合约会到期，不能天然形成全年曲线；本 Notebook 每个交易日重新选择同一规则的代表合约，展示最活跃 Call/Put 和 0DTE ATM Call/Put 的滚动序列。Notebook 不调用 Massive API，也不读取 Source gzip。

In [ ]:
from pathlib import Path
from decimal import Decimal
import json, os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'kairospy').exists(): ROOT = ROOT.parent
DATASET_ID = os.environ.get('SPXW_DAY_AGG_DATASET', 'options.us.massive.spxw.day-aggs.2026-ytd.v2')
DATASET_ROOT = ROOT / 'data/curated/provider=massive' / f'dataset={DATASET_ID}'
if not DATASET_ROOT.exists(): raise FileNotFoundError(f'请先运行 prepare-spxw-daily-ohlcv；缺少 {DATASET_ROOT}')
print({'dataset': DATASET_ID, 'directory': str(DATASET_ROOT)})

## 1. Dataset 血缘、完整性和覆盖范围

In [ ]:
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text())
quality = json.loads((DATASET_ROOT / 'quality.json').read_text())
coverage = json.loads((DATASET_ROOT / 'coverage.json').read_text())
lineage = json.loads((DATASET_ROOT / 'lineage.json').read_text())
assert quality['publishable'] and quality['missing_trading_days'] == []
assert lineage['provider'] == 'massive' and lineage['api_base'] == 'https://api.massiveprivateserver.site'
display(pd.DataFrame([{
    'dataset_id': DATASET_ID, 'start': coverage['start'], 'end_exclusive': coverage['end'],
    'trading_days': coverage['trading_days'], 'spxw_rows': manifest['rows'],
    'inventory_hash': manifest['inventory_sha256'], 'dataset_hash': manifest['dataset_sha256'],
    'publishable': quality['publishable'],
}]))

## 2. 月度 Parquet 与具体合约热门榜

具体 ticker 热门榜用于审计当日/到期合约活跃度。由于 ticker 包含到期日，它不应被解释为全年连续品种。

In [ ]:
import pyarrow.dataset as ds
monthly_files = [DATASET_ROOT / item['path'] for item in manifest['files'] if item.get('month')]
table = ds.dataset(monthly_files, format='parquet').to_table()
daily = table.to_pandas()
for column in ('strike','open','high','low','close','volume'): daily[column] = daily[column].map(float)
daily['event_date'] = pd.to_datetime(daily['event_date'])
daily['expiry'] = pd.to_datetime(daily['expiry'])
ranking = (daily.groupby(['ticker','right','strike','expiry'], as_index=False)
           .agg(volume=('volume','sum'), transactions=('transactions','sum'), active_days=('event_date','nunique'), average_close=('close','mean'))
           .sort_values(['volume','transactions','active_days'], ascending=False))
display(ranking.head(20))

## 3. 每日滚动代表合约

- `top_call/top_put`：当天 volume 最大，transactions 和 ticker 作为稳定次级排序。
- `atm_0dte_call/put`：当天到期、同执行价 Call/Put 的 close 用 `F=K+C-P` 推导合成远期，再选择最接近该远期的执行价。
- 每一行都是一个交易日，因此下面的曲线具有 2026 YTD 的连续日度横轴；ticker 会随日期滚动。

In [ ]:
import pyarrow.parquet as pq
rolling = pq.read_table(DATASET_ROOT / 'daily_representatives.parquet').to_pandas()
rolling['event_date'] = pd.to_datetime(rolling['event_date'])
numeric = [column for column in rolling.columns if column.endswith(('_close','_volume','_strike')) or column in ('synthetic_forward_0dte','total_volume')]
for column in numeric: rolling[column] = rolling[column].map(lambda value: float(value) if value is not None else None)
display(rolling.head())
print({'rows': len(rolling), 'first': rolling.event_date.min(), 'last': rolling.event_date.max()})

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
axes[0].plot(rolling.event_date, rolling.total_volume, label='total SPXW volume')
axes[0].set_title('1. Daily SPXW aggregate volume'); axes[0].set_ylabel('contracts'); axes[0].legend()
axes[1].plot(rolling.event_date, rolling.active_contracts, label='active contracts', color='tab:purple')
axes[1].set_title('2. Active SPXW contracts per day'); axes[1].set_ylabel('contracts'); axes[1].legend()
axes[2].plot(rolling.event_date, rolling.top_call_close, label='daily most-active call close')
axes[2].plot(rolling.event_date, rolling.top_put_close, label='daily most-active put close')
axes[2].set_title('3. Rolling most-active Call / Put close'); axes[2].set_ylabel('option points'); axes[2].legend()
axes[3].plot(rolling.event_date, rolling.atm_0dte_call_close, label='0DTE ATM call close')
axes[3].plot(rolling.event_date, rolling.atm_0dte_put_close, label='0DTE ATM put close')
axes[3].set_title('4. Rolling 0DTE ATM Call / Put close'); axes[3].set_ylabel('option points'); axes[3].legend()
axes[3].set_xlabel('trading date'); plt.tight_layout(); plt.show()

## 4. 滚动选择审计

下面保留每天实际选择的 ticker、执行价、价格和成交量，确保连续曲线没有隐藏换月/换合约。

In [ ]:
columns = ['event_date','synthetic_forward_0dte','top_call_ticker','top_call_close','top_call_volume','top_put_ticker','top_put_close','top_put_volume','atm_0dte_call_ticker','atm_0dte_call_close','atm_0dte_put_ticker','atm_0dte_put_close']
display(rolling[columns].tail(20))

## 使用边界

- Day Aggregates 是成交 OHLC，不是 bid/ask；不能直接证明可执行价格。
- `top_*` 是每日重新选择的滚动研究序列，不是可直接持有一整年的单一合约。
- 0DTE 合成远期使用日收盘 Call/Put 配对，仅用于日度观察；正式盘中回测仍使用 point-in-time Quotes。
- 当前 Dataset 覆盖 2026-01-02 至 2026-07-14，是截至最近已下载完整交易日的 YTD，而不是尚未发生的全年。